# Bab 22: Deteksi Anomali

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab22/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab22")
print("Folder kerja:", os.getcwd())

## Listing 22.1: Diagram kendali individual durasi sangrai

`kode/bab22/shewhart.py`

In [ ]:
import numpy as np
import pandas as pd

b = pd.read_csv("kode/data/batch_sangrai.csv")
x = b.durasi_detik.to_numpy(float)
fase1 = x[:30]
mr = np.abs(np.diff(fase1))
pusat = fase1.mean()
sigma = mr.mean() / 1.128
bka, bkb = pusat + 3 * sigma, pusat - 3 * sigma
print(f"fase I (30 batch): rata-rata {pusat:.1f} s, "
      f"sigma (MR/1,128) {sigma:.1f} s")
print(f"batas kendali: {bkb:.1f} - {bka:.1f} detik")
luar = np.where((x > bka) | (x < bkb))[0]
for i in luar:
    print(f"  {b.id_batch[i]}: durasi {x[i]:.0f} s"
          f"  ({b.catatan_operator[i]})")

z = (x - pusat) / sigma
for i in range(8, len(z)):
    if (z[i - 8:i + 1] > 0).all() or (z[i - 8:i + 1] < 0).all():
        print(f"  aturan 9 berturut-turut sepihak: berakhir di "
              f"{b.id_batch[i]}")
        break
else:
    print("  aturan 9 berturut-turut sepihak: tidak ada sinyal")
with open("gambar/bab22/shewhart.dat", "w") as f:
    f.write("i x\n")
    for i, v in enumerate(x, 1):
        f.write(f"{i} {v:.0f}\n")

## Listing 22.2: Shewhart, CUSUM, dan EWMA untuk struk batal

`kode/bab22/cusum_ewma.py`

In [ ]:
import numpy as np
import pandas as pd

a = pd.read_csv("kode/data/audit_kasir.csv", parse_dates=["tanggal"])
MULAI = pd.Timestamp("2026-02-01")
jan = a[a.tanggal < MULAI]
P_SEMUA = jan.struk_batal.sum() / jan.jumlah_struk.sum()

def pantau(kid, k=1.0, h=5.0, lam=0.1, L=3.0):
    s = a[a.id_kasir == kid].set_index("tanggal")
    f1 = s[s.index < MULAI]                            # fase I
    p0 = (f1.struk_batal.sum() + 1000 * P_SEMUA) \
        / (f1.jumlah_struk.sum() + 1000)
    s = s[s.index >= MULAI]
    n = s.jumlah_struk
    z = (s.struk_batal - n * p0) / np.sqrt(n * p0 * (1 - p0))
    c = e = 0.0
    batas_e = L * np.sqrt(lam / (2 - lam))
    sinyal, jejak = {}, []
    for hari, zt in zip((s.index - MULAI).days + 1, z.to_numpy()):
        c = max(0.0, c + zt - k)
        e = lam * zt + (1 - lam) * e
        jejak.append((hari, zt, c, e))
        for nama, ya in [("Shewhart", zt > 3), ("CUSUM", c > h),
                         ("EWMA", e > batas_e)]:
            if ya and nama not in sinyal:
                sinyal[nama] = hari
    return sinyal, jejak

print(f"p batal rata-rata Januari: {P_SEMUA:.4f}")
print("hari ke- sinyal pertama sejak 1 Februari:")
for kid in ["G07-K2", "G31-K3"]:
    sinyal, jejak = pantau(kid)
    teks = [f"{n} {sinyal.get(n, '-')}"
            for n in ["Shewhart", "CUSUM", "EWMA"]]
    print(f"  {kid}: " + ", ".join(teks))
    if kid == "G07-K2":
        with open("gambar/bab22/cusum.dat", "w") as f:
            f.write("hari z c e\n")
            for hari, zt, c, e in jejak:
                f.write(f"{hari} {zt:.3f} {c:.3f} {e:.3f}\n")

alarm = {"Shewhart": 0, "CUSUM": 0, "EWMA": 0}
jujur = [k for k in a.id_kasir.unique()
         if k not in ("G07-K2", "G31-K3", "G18-K1")]
for kid in jujur:
    for nama in pantau(kid)[0]:
        alarm[nama] += 1
print(f"kasir jujur yang memicu alarm Feb-Mar (dari {len(jujur)}):")
print("  " + ", ".join(f"{n} {v}" for n, v in alarm.items()))

## Listing 22.3: Diagram $T^2$ Hotelling untuk batch sangrai

`kode/bab22/hotelling.py`

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

b = pd.read_csv("kode/data/batch_sangrai.csv")
KOL = ["berat_kg", "durasi_detik", "suhu_akhir", "detik_crack"]
X = b[KOL].to_numpy(float)
f1 = X[:30]
m, p = len(f1), len(KOL)
mu = f1.mean(0)
S = np.cov(f1, rowvar=False)
Sinv = np.linalg.inv(S)
d = X - mu
T2 = np.einsum("ij,jk,ik->i", d, Sinv, d)
bka = p * (m + 1) * (m - 1) / (m * (m - p)) \
    * stats.f.ppf(0.9973, p, m - p)
print(f"batas kendali T2 (fase II, alfa 0,0027): {bka:.2f}")
urut = np.argsort(-T2)[:6]
print("enam T2 terbesar:")
for i in urut:
    z = d[i] / np.sqrt(np.diag(S))
    j = int(np.argmax(np.abs(z)))
    tanda = "SINYAL" if T2[i] > bka else "      "
    print(f"  {b.id_batch[i]} T2 {T2[i]:5.1f} {tanda} terbesar:"
          f" {KOL[j]} (z {z[j]:+.1f})")

r = np.corrcoef(f1, rowvar=False)
print(f"korelasi durasi-crack fase I: {r[1, 3]:.2f}")
with open("gambar/bab22/t2.dat", "w") as f:
    f.write("i t2\n")
    for i, v in enumerate(T2, 1):
        f.write(f"{i} {v:.2f}\n")

## Listing 22.4: Lonjakan sensor dan pita profil RoR

`kode/bab22/sensor.py`

In [ ]:
import numpy as np
import pandas as pd

s = pd.read_csv("kode/data/sensor_sangrai.csv")
b = pd.read_csv("kode/data/batch_sangrai.csv").set_index("id_batch")

def hampel(x, w=7, t=6.0, minimal=5.0):
    """Tandai titik yang jauh dari median jendela (dalam MAD)."""
    x = pd.Series(x)
    med = x.rolling(w, center=True, min_periods=1).median()
    mad = (x - med).abs().rolling(w, center=True,
                                  min_periods=1).median()
    batas = np.maximum(t * 1.4826 * mad, minimal)
    return ((x - med).abs() > batas).to_numpy()

def ror(x, w=30):
    """Laju kenaikan suhu (derajat per menit), dihaluskan."""
    x = pd.Series(x).rolling(9, center=True, min_periods=1).median()
    return (x.diff(w) * 60 / w).to_numpy()

kurva = {k: g.suhu_biji.to_numpy() for k, g in s.groupby("id_batch")}
T = 480
acuan = np.array([ror(kurva[k])[200:T] for k in b.index[:30]
                  if b.catatan_operator[k] == "normal"
                  and not hampel(kurva[k]).any()])
mu, sd = acuan.mean(0), acuan.std(0)
print(f"profil acuan RoR dari {len(acuan)} batch fase I")
print(f"RoR rata-rata pada detik 300: {mu[100]:.1f} C/menit")

baris = []
for k, x in kurva.items():
    lonjak = int(hampel(x).sum())
    zr = (ror(x)[200:T] - mu) / sd
    lama = int((np.abs(zr) > 3).sum())                 # detik
    baris.append((k, lonjak, lama, b.catatan_operator[k]))
h = pd.DataFrame(baris, columns=["batch", "lonjakan", "detik_luar",
                                 "catatan"]).set_index("batch")
print("\nbatch bersinyal (ada lonjakan, atau RoR di luar pita")
print("lebih dari 30 detik):")
print(h[(h.lonjakan > 0) | (h.detik_luar > 30)].to_string())
lain = h.drop(["B017", "B071", "B038", "B055"])
print(f"\n76 batch lain: lonjakan maks {lain.lonjakan.max()}, "
      f"detik luar pita maks {lain.detik_luar.max()}")

with open("gambar/bab22/ror.dat", "w") as f:
    f.write("t lo hi b038 b055 b001\n")
    r38, r55 = ror(kurva["B038"]), ror(kurva["B055"])
    r01 = ror(kurva["B001"])
    for i in range(200, T, 2):
        f.write(f"{i} {mu[i - 200] - 3 * sd[i - 200]:.2f} "
                f"{mu[i - 200] + 3 * sd[i - 200]:.2f} {r38[i]:.2f} "
                f"{r55[i] if i < len(r55) else float('nan'):.2f} "
                f"{r01[i]:.2f}\n")
with open("gambar/bab22/lonjak.dat", "w") as f:
    f.write("t x tanda\n")
    x = kurva["B017"]; m = hampel(x)
    for i in range(0, len(x), 1):
        f.write(f"{i} {x[i]:.1f} {int(m[i])}\n")

## Listing 22.5: Studi kasus audit kasir

`kode/bab22/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler

a = pd.read_csv("kode/data/audit_kasir.csv", parse_dates=["tanggal"])
a["rasio_batal"] = a.struk_batal / a.jumlah_struk
a["diskon_per_struk"] = a.diskon_manual / a.jumlah_struk
FITUR = ["rasio_batal", "diskon_per_struk", "porsi_tunai",
         "struk_setelah_tutup"]
X = RobustScaler().fit_transform(a[FITUR])

CURANG = {"G07-K2": "2026-02-01", "G31-K3": "2026-02-01",
          "G18-K1": "2026-01-20"}
y = np.array([k in CURANG and t >= pd.Timestamp(CURANG[k])
              for k, t in zip(a.id_kasir, a.tanggal)])
print(f"{len(a)} kasir-hari, {y.sum()} di antaranya curang "
      f"({y.mean():.1%})")

skor = {
    "z robust (maks)": np.abs(X).max(1),
    "LOF": -LocalOutlierFactor(n_neighbors=50).fit(X)
            .negative_outlier_factor_,
    "Isolation Forest": -IsolationForest(n_estimators=300,
                                         random_state=0).fit(X)
                         .score_samples(X),
}
print("\ntingkat kasir-hari:    AUC    AP")
for nama, s in skor.items():
    print(f"  {nama:17s} {roc_auc_score(y, s):.3f}  "
          f"{average_precision_score(y, s):.3f}")

a["skor"] = skor["Isolation Forest"]
a["minggu"] = a.tanggal.dt.to_period("W-SUN")
mingguan = a.groupby(["minggu", "id_kasir"]).skor.mean().reset_index()
print("\ndaftar audit mingguan: 10 kasir dengan skor rata-rata")
print("tertinggi; berapa minggu kasir curang masuk daftar?")
for k, t in CURANG.items():
    mg = mingguan[mingguan.minggu.dt.end_time >= pd.Timestamp(t)]
    masuk = sum(k in g.nlargest(10, "skor").id_kasir.tolist()
                for _, g in mg.groupby("minggu"))
    print(f"  {k}: {masuk} dari {mg.minggu.nunique()} minggu")
daftar = [k for _, g in mingguan.groupby("minggu")
          for k in g.nlargest(10, "skor").id_kasir if k not in CURANG]
hit = pd.Series(daftar).value_counts()
print(f"kasir jujur yang pernah masuk daftar: {hit.size} dari 117")
print(f"paling sering: {hit.index[0]} ({hit.iloc[0]} minggu)")

feb = a[a.tanggal >= "2026-02-01"]
prof = feb.groupby("id_kasir")[FITUR].mean()
ref = prof.median()
print("\nprofil kasir yang ditandai vs median semua kasir (Feb-Mar):")
print(pd.concat([prof.loc[list(CURANG)], ref.rename("median")
                 .to_frame().T]).round(3).to_string())